<a href="https://colab.research.google.com/github/carlosaupta-blip/Esp32_Yolo26_/blob/Entrenamiento_modelo/quantizar_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalar dependencias


In [ ]:
%pip install -q 'numpy<2.0.0'
%pip install -q esp-ppq==1.3.11
%pip install -q onnx==1.17.0
%pip install -q 'onnxruntime>=1.19.0'
%pip install -q 'torch>=2.4.0'
%pip install -q 'torchvision>=0.19.0'
%pip install -q 'ultralytics==8.4.7'
%pip install -q 'onnxsim>=0.4.36'
%pip install -q roboflow

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, b

🧱 Step 1: Download & Prepare LEGO Minifigures Dataset

In [ ]:
from roboflow import Roboflow
from google.colab import drive
drive.mount('/content/drive')
import os
from dotenv import load_dotenv

# Specify the path to your .env file in Google Drive
dotenv_path = '/content/drive/MyDrive/Colab Notebooks/.env'

# Load environment variables from the .env file
if os.path.exists(dotenv_path):
    load_dotenv(dotenv_path) # This loads variables into os.environ
    print(f"Loaded .env file from {dotenv_path}")
    # Retrieve the API key from the environment variables
    roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")
else:
    print(f"Warning: .env file not found at {dotenv_path}. Please create one with your ROBOFLOW_API_KEY.")
    # Fallback if .env not found, or if API key isn't in .env
    roboflow_api_key = os.getenv("ROBOFLOW_API_KEY", "YOUR_API_KEY_HERE")

# Ensure roboflow_api_key is a string, even if .env was missing or didn't contain it
if not isinstance(roboflow_api_key, str) or roboflow_api_key == "YOUR_API_KEY_HERE":
    print("Error: Roboflow API key not found or is default. Please set ROBOFLOW_API_KEY in your .env file or hardcode it.")
    # You might want to raise an exception or exit here if the key is critical
    # For now, we'll continue with the placeholder, which will likely fail the Roboflow call


rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("carlos-aponte-upta").project("face-recognition-abgq6-yav2q")
version = project.version(1)
dataset = version.download("yolo26")


Mounted at /content/drive
Loaded .env file from /content/drive/MyDrive/Colab Notebooks/.env
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Face-recognition-1 in yolo26:: 100%|██████████| 32191/32191 [00:09<00:00, 3495.84it/s]


Step 2: Fix Absolute Paths in data.yaml

In [ ]:
import yaml
import os

# dataset.location comes from the previous cell (NuJ6TQkbHIt4)
yaml_path = os.path.join(dataset.location, 'data.yaml')

# 1. Read the current yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# 2. Fix the path logic
data['path'] = dataset.location # Use the actual downloaded path
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

# 3. Write it back
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml paths updated!")
print(f"New 'path' is: {data['path']}")
DRIVE_SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO26'

✅ data.yaml paths updated!
New 'path' is: /content/Face-recognition-1


Step 3: Fine-Tuning YOLO26n

In [ ]:
from ultralytics import YOLO
import os
import torch

# Disable unwanted loggers
os.environ["COMET_MODE"] = "disabled"
os.environ["WANDB_MODE"] = "disabled"


# Determine the device to use
device = 0 if torch.cuda.is_available() else 'cpu' # Use GPU 0 if available, else CPU
print(f"Using device: {device}")

# Load pre-trained base model
model = YOLO('yolo26n.pt')

# Fine-tune on LEGO dataset
results = model.train(
    data=yaml_path,
    epochs=15,
    imgsz=512,
    batch=24,
    device=device,
    project=os.path.dirname(DRIVE_SAVE_PATH), # Carpeta padre
    name=os.path.basename(DRIVE_SAVE_PATH),    # Nombre de la carpeta del experimento
    plots=True,

    # MuSGD Fine-tuning setup
    optimizer='MuSGD',
    lr0=0.005,
    lrf=0.01,
)

print("✅ Training complete!")

Using device: 0
New https://pypi.org/project/ultralytics/8.4.154 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.7 🚀 Python-3.12.13 torch-2.9.1+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=24, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Face-recognition-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Entren

KeyboardInterrupt: 

In [ ]:
import shutil
from glob import glob

# Get the most recently modified best.pt weights trace
weight_files = glob("/content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO262/weights/best.pt")

if weight_files:
    latest_weights = max(weight_files, key=os.path.getctime)
    destination = "yolo26n_face.pt"
    shutil.copy(latest_weights, destination)
    print(f"✅ Success! Extracted Custom weights as: {os.path.abspath(destination)}")
else:
    print("❌ Error: Could not find 'best.pt' in runs/detect/. Did training finish?")

✅ Success! Extracted Custom weights as: /content/yolo26n_face.pt


🛠️ Step 4: YOLOv26 Output Quantization Pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive') # Asegurarse de que Drive esté montado

import os
import shutil

repo_url = 'https://github.com/espressif/esp-dl.git'
repo_name = 'esp-dl-repo'
clone_path = f'/content/{repo_name}'

# Clonar el repositorio si no ha sido clonado ya
if not os.path.exists(clone_path):
    print(f'Clonando el repositorio {repo_url}...')
    !git clone {repo_url} {clone_path}
else:
    print(f'El repositorio ya está clonado en {clone_path}. Saltando el clonado.')

# Ruta de origen de la carpeta 'scripts' dentro del repositorio clonado
source_scripts_path = os.path.join(clone_path, 'examples/tutorial/how_to_quantize_model/quantize_yolo26/scripts')
# Ruta de destino para la carpeta 'scripts' en la raíz de Colab
destination_scripts_path = '/content/scripts'

# Copiar la carpeta 'scripts' al directorio raíz de Colab si no existe
if os.path.exists(source_scripts_path) and not os.path.exists(destination_scripts_path):
    print(f'Copiando la carpeta "scripts" de {source_scripts_path} a {destination_scripts_path}...')
    shutil.copytree(source_scripts_path, destination_scripts_path)
    print('Carpeta "scripts" copiada exitosamente.')
elif os.path.exists(destination_scripts_path):
    print('La carpeta "scripts" ya existe en /content/. Saltando la copia.')
else:
    print(f'Error: No se encontró la carpeta "scripts" en la ruta esperada: {source_scripts_path}')

# Verificar el contenido de la carpeta 'scripts' para confirmación
if os.path.exists(destination_scripts_path):
    print('\nContenido de /content/scripts:')
    for item in os.listdir(destination_scripts_path):
        print(f'  - {item}')
else:
    print('\nLa carpeta /content/scripts NO EXISTE después de intentar copiarla.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clonando el repositorio https://github.com/espressif/esp-dl.git...
Cloning into '/content/esp-dl-repo'...
remote: Enumerating objects: 17698, done.
remote: Counting objects: 100% (1243/1243), done.
remote: Compressing objects: 100% (298/298), done.
remote: Total 17698 (delta 1022), reused 985 (delta 941), pack-reused 16455 (from 3)
Receiving objects: 100% (17698/17698), 822.06 MiB | 23.08 MiB/s, done.
Resolving deltas: 100% (9983/9983), done.
Updating files: 100% (1337/1337), done.
Copiando la carpeta "scripts" de /content/esp-dl-repo/examples/tutorial/how_to_quantize_model/quantize_yolo26/scripts a /content/scripts...
Carpeta "scripts" copiada exitosamente.

Contenido de /content/scripts:
  - esp_ppq_patch.py
  - __init__.py
  - export.py
  - notebook_helpers.py
  - trainer.py
  - dataset.py
  - utils.py
  - validator.py
  - esp_ppq_patch_2.py


In [ ]:
import os
import sys

sys.path.append('/content/scripts')

import torch
import types
from esp_ppq.api import get_target_platform
import esp_ppq.lib as PFL
from esp_ppq.executor import TorchExecutor
from esp_ppq.core import QuantizationVisibility, TargetPlatform
from esp_ppq.api.interface import load_onnx_graph
from esp_ppq.quantization.optim import (
    QuantizeSimplifyPass, QuantizeFusionPass, ParameterQuantizePass,
    RuntimeCalibrationPass, PassiveParameterQuantizePass, QuantAlignmentPass,
    LearnedStepSizePass
)


    ___________ ____        ____  ____  ____
   / ____/ ___// __ \      / __ \/ __ \/ __ \
  / __/  \__ \/ /_/ /_____/ /_/ / /_/ / / / /
 / /___ ___/ / ____/_____/ ____/ ____/ /_/ /
/_____//____/_/         /_/   /_/    \___\_\




In [ ]:
IMG_SZ_I = 512
PLATFORM = "s3"
PROJECT_NAME = "face"
DATA_YAML_FILE_I = "/content/Face-recognition-1/data.yaml"
INT16_LUT_STEP_I = 32

In [ ]:
class QATConfig:
    IMG_SZ = IMG_SZ_I
    DEVICE = "cuda" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"
    DATA_YAML_FILE = DATA_YAML_FILE_I
    BATCH_SIZE = 16
    CALIB_MAX_IMAGES = 1200
    CALIB_VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    DATA_FALLBACK_PATH = "/content/drive/MyDrive/Colab Notebooks/Imagenes de calibracion"

    CALIB_STEPS = 16
    QUANT_CALIB_METHOD = "percentile"
    QUANT_ALIGNMENT = "Align to Output"
    TARGET_PLATFORM = get_target_platform("esp32" + PLATFORM, 8)

    INT16_LUT_STEP = INT16_LUT_STEP_I

    # Streaming specific: chunk size for TCN/Streaming inference
    # Typically 16 or 32 depending on ESP32-P4 memory constraints
    STREAMING_CHUNK = None

    # TQT Specific Hyperparameters
    TQT_STEPS = 50
    TQT_LR = 1e-5
    TQT_INT_LAMBDA = 0.25
    TQT_BLOCK_SIZE = 8
    TQT_COLLECTING_DEVICE = "cpu"

    BASE_DIR = os.getcwd()
    MODEL_NAME = f"yolo26n_{PROJECT_NAME}"
    PT_FILE = f"{MODEL_NAME}.pt"
    ONNX_FILE = f"{MODEL_NAME}_export.onnx"

    ESPDL_OUTPUT_DIR = os.path.join(BASE_DIR, "output", f"{PROJECT_NAME}_{IMG_SZ_I}_s8_{PLATFORM}")
    ONNX_PATH = os.path.join(ESPDL_OUTPUT_DIR, ONNX_FILE)

if 'config' not in sys.modules:
    sys.modules['config'] = types.ModuleType('config')
sys.modules['config'].QATConfig = QATConfig

In [ ]:
from utils import seed_everything, register_mod_op, get_exclusive_ancestors
from dataset import get_calibration_loader
from ultralytics.data.utils import check_det_dataset
from esp_ppq_patch import apply_esp_ppq_patches
from esp_ppq_patch_2 import apply_addlut_patch
from notebook_helpers import extract_model_meta, prepare_onnx, prune_graph_safely

%cd /content/esp-dl-repo/examples/tutorial/how_to_quantize_model/quantize_yolo26

from esp_ppq_lut.passes import EspdlLUTFusionPass
from esp_ppq_lut.exporter import HardwareAwareEspdlExporter

os.makedirs(QATConfig.ESPDL_OUTPUT_DIR, exist_ok=True)

import esp_ppq_lut as esp_lut
esp_lut.initialize(step=QATConfig.INT16_LUT_STEP, verbose=True)

seed_everything(1234)
register_mod_op()
apply_esp_ppq_patches()
apply_addlut_patch()

print("Environment and Configuration setup complete.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
/content/esp-dl-repo/examples/tutorial/how_to_quantize_model/quantize_yolo26
[ESP-PPQ-LUT] Activation forwarders registered for simulation.
[ESPDL Emulator] LUT Operation Handler Registered Globally.
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_INT8
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_INT16
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_S3_INT8
[ESPDL Exporter] Registered HardwareAwareExporter for: ESPDL_S3_INT16
[ESP-PPQ-LUT] Activation forwarders registered for simulation.
[ESP-PPQ-LUT] Extension Initialized (Default Step=32)
Registered 'Mod' handler for PPQ.
Applying ESP-PPQ Runtime Patches...
  [x] Patched OnnxParser.refin

In [ ]:
import sys
%pip install onnxscript

# Actualizar PT_FILE a la ruta absoluta correcta
sys.modules['config'].QATConfig.PT_FILE = "/content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO262/weights/best.pt"

# Asegurarse de que el modelo ONNX se exporta con opset=18 o superior
prepare_onnx()
model_meta = extract_model_meta()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 18.5 MB/s eta 0:00:00
Applying ESP-DL patches for export...
Patched 2 Attention modules.
Patched Detect module: <class 'ultralytics.nn.modules.head.Detect'>
Ultralytics 8.4.7 🚀 Python-3.12.13 torch-2.9.1+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
>> Fuse method blocked! Keeping all heads.
YOLO26n summary (fused): 146 layers, 2,494,694 parameters, 0 gradients, 5.6 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO262/weights/best.pt' with input shape (16, 3, 512, 512) BCHW and output shape(s) ((16, 5, 64, 64), (16, 5, 32, 32), (16, 5, 16, 16), (16, 5, 64, 64), (16, 5, 32, 32), (16, 5, 16, 16)) (5.1 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempti

W0918 04:05:56.594000 288 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0918 04:05:57.547000 288 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0918 04:05:57.548000 288 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: '

ONNX: simplifying with onnxsim v0.7.3...
ONNX: export success ✅ 15.3s, saved as '/content/output/face_512_s8_s3/yolo26n_face_export.onnx' (9.6 MB)

Export complete (19.8s)
Results saved to /content/drive/MyDrive/Colab Notebooks/EntrenamientoYOLO262/weights
Predict:         yolo predict task=detect model=/content/output/face_512_s8_s3/yolo26n_face_export.onnx imgsz=512 
Validate:        yolo val task=detect model=/content/output/face_512_s8_s3/yolo26n_face_export.onnx imgsz=512 data=/content/Face-recognition-1/data.yaml  
Visualize:       https://netron.app
Exported base ONNX to /content/output/face_512_s8_s3/yolo26n_face_export.onnx
Metadata: NC=1, RegMax=1, Stride=[8.0, 16.0, 32.0]


In [ ]:
print("Loading ONNX Graph into ESP-PPQ...")
graph = load_onnx_graph(onnx_import_file=QATConfig.ONNX_PATH)

output_names = list(graph.outputs.keys())
aux_ops = set()
main_ops = set()

if len(output_names) >= 6:
    aux_outputs = output_names[0:3]
    main_outputs = output_names[3:6]
    aux_ops = get_exclusive_ancestors(graph, aux_outputs, main_outputs)
    main_ops = get_exclusive_ancestors(graph, main_outputs, aux_outputs)

quantizer = PFL.Quantizer(platform=QATConfig.TARGET_PLATFORM, graph=graph)
dispatching_table = PFL.Dispatcher(graph=graph, method="conservative").dispatch(
    quantizer.quant_operation_types
)

for opname, platform in dispatching_table.items():
    if platform == TargetPlatform.UNSPECIFIED:
        dispatching_table[opname] = TargetPlatform(quantizer.target_platform)

for op in aux_ops:
    if op.name in dispatching_table:
        dispatching_table[op.name] = TargetPlatform.FP32

Loading ONNX Graph into ESP-PPQ...


In [ ]:
INT16_PLATFORM = get_target_platform("esp32" + PLATFORM, 16)

# Force high-sensitivity exit layers to INT16
int16_layers = {
    # Neck Exits
    "/model.16/cv2/conv/Conv", "/model.16/cv2/conv/Conv/Swish",
    "/model.19/cv2/conv/Conv", "/model.19/cv2/conv/Conv/Swish",
    "/model.22/cv2/conv/Conv", "/model.22/cv2/conv/Conv/Swish",

    # Box Heads
    "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv",
    "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv",
    "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv",

    # Class Heads
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv"
}

for op in graph.operations.values():
    if op.name in dispatching_table and op.name in int16_layers:
        dispatching_table[op.name] = INT16_PLATFORM

# Workaround FP32 Concat Limits
fp32_layers = {"/model.23/Concat_5", "/model.23/Concat_3", "/model.23/Concat_4"}
for op in main_ops:
    if op.name in fp32_layers:
        dispatching_table[op.name] = TargetPlatform.FP32

print("Applying Dispatcher Types...")
for op in graph.operations.values():
    quantizer.quantize_operation(op_name=op.name, platform=dispatching_table[op.name])

Applying Dispatcher Types...


In [ ]:
# The RuntimeError in this cell, which prevents the model from being saved, is likely due to a malformed
# ONNX graph resulting from an upstream error in cell gIZyQNxdHuCP.
# The error in gIZyQNxdHuCP indicated 'No Adapter To Version $17 for Resize' when converting to opset 13.
# To fix this, you need to modify cell gIZyQNxdHuCP to ensure the ONNX model is exported with opset=18.
# For example, by changing `prepare_onnx()` to `prepare_onnx(opset=18)` if it supports the argument,
# or by directly modifying the `model.export()` call within `notebook_helpers.py` to use `opset=18`.

import sys
# Importar exporter para guardar el modelo
from esp_ppq_lut.exporter import HardwareAwareEspdlExporter
import os # Asegurarse de que os esté importado para os.makedirs, os.path.join

print(f"Running Optimization Pipeline (Streaming: {QATConfig.STREAMING_CHUNK is not None})...")
# DRIVE_SAVE_PATH se define globalmente en AqV2d9C0HT2H, no es necesario redefinirlo aquí.
data_cfg = check_det_dataset(QATConfig.DATA_YAML_FILE)
cali_loader = get_calibration_loader(data_cfg)

executor = TorchExecutor(graph=graph, device=QATConfig.DEVICE) # Explicitly pass device
dummy_input = torch.zeros([QATConfig.BATCH_SIZE, 3, QATConfig.IMG_SZ, QATConfig.IMG_SZ]).to(QATConfig.DEVICE) # Use QATConfig.BATCH_SIZE
executor.tracing_operation_meta(inputs=dummy_input)

pipeline = PFL.Pipeline([
    QuantizeSimplifyPass(),
    QuantizeFusionPass(activation_type=quantizer.activation_fusion_types),
    ParameterQuantizePass(),
    RuntimeCalibrationPass(method=QATConfig.QUANT_CALIB_METHOD),

    # Accuracy Recovery
    LearnedStepSizePass(
        steps=QATConfig.TQT_STEPS,
        lr=QATConfig.TQT_LR,
        block_size=QATConfig.TQT_BLOCK_SIZE,
        collecting_device=QATConfig.DEVICE # Changed from QATConfig.TQT_COLLECTING_DEVICE to QATConfig.DEVICE
    ),

    PassiveParameterQuantizePass(clip_visiblity=QuantizationVisibility.EXPORT_WHEN_ACTIVE),
    QuantAlignmentPass(elementwise_alignment=QATConfig.QUANT_ALIGNMENT),

    # Direct LUT Conversion for HW int16 Swish emulation
    EspdlLUTFusionPass(
        target_ops=['Swish'],
        lut_step=QATConfig.INT16_LUT_STEP
    )
])

try:
    pipeline.optimize(
        calib_steps=QATConfig.CALIB_STEPS,
        collate_fn=(lambda x: x.type(torch.float).to(QATConfig.DEVICE)),
        graph=graph,
        dataloader=cali_loader,
        executor=executor,
    )

    # ========================================== # Recorte y estructuración de salidas para ESP-DL # ========================================== print("Dividiendo nodos Concat de salida en 6 tensores discretos...") # 1\. Remover cabezas auxiliares si existen output\_names = list(graph.outputs.keys()) if len(output\_names) &gt;= 6: for name in output\_names[0:3]: if name in graph.outputs: graph.outputs.pop(name) prune\_graph\_safely(graph) # 2\. Dividir el Concat en cajas (Box) y clases (Cls) targets = ["one2one\_p3", "one2one\_p4", "one2one\_p5"] collected\_outputs = {} for target\_name in targets: if target\_name in graph.outputs: original\_output\_var = graph.variables[target\_name] producer = original\_output\_var.source\_op if producer and producer.type == "Concat": box\_var, cls\_var = None, None for input\_var in producer.inputs: dims = input\_var.shape if dims is not None: if 4 in dims: box\_var = input\_var elif model\_meta['nc'] in dims: cls\_var = input\_var if box\_var and cls\_var: pair\_config = [ (box\_var, f"{target\_name}\_box"), (cls\_var, f"{target\_name}\_cls"), ] for var, new\_name in pair\_config: old\_name = var.name if old\_name in graph.variables: graph.variables.pop(old\_name) var.\_name = new\_name graph.variables[new\_name] = var collected\_outputs[new\_name] = var graph.outputs.pop(target\_name) graph.remove\_operation(producer, keep\_coherence=False) for var in producer.inputs: if producer in var.dest\_ops: var.dest\_ops.remove(producer) # 3\. Establecer el orden exacto esperado por el firmware C++ de ESP-DL final\_output\_list = [ "one2one\_p3\_box", "one2one\_p3\_cls", "one2one\_p4\_box", "one2one\_p4\_cls", "one2one\_p5\_box", "one2one\_p5\_cls" ] graph.outputs.clear() for name in final\_output\_list: if name in collected\_outputs: graph.outputs[name] = collected\_outputs[name] prune\_graph\_safely(graph) print("✅ Salidas del grafo recortadas y estructuradas correctamente.")

except RuntimeError as e:
    print(f"❌ ERROR: A RuntimeError occurred during the quantization pipeline: {e}")
    print("This is likely due to the malformed ONNX graph from the previous step (cell gIZyQNxdHuCP).")
    print("Please address the ONNX export error in cell gIZyQNxdHuCP first.")
except Exception as e:
    print(f"❌ ERROR: An unexpected error occurred: {e}")
    print("Please review the previous steps for potential issues.")

Running Optimization Pipeline (Streaming: False)...
Using dataset at: /content/Face-recognition-1/train/images
[WARNING][PPQ][2026-09-18 04:06:08]:  Unexpected input value of operation node_upsample_nearest2d, recieving "None" at its input 1
[WARNING][PPQ][2026-09-18 04:06:08]:  Unexpected input value of operation node_upsample_nearest2d_1, recieving "None" at its input 1
[04:06:08] PPQ Quantize Simplify Pass Running ...         Finished.
[04:06:08] PPQ Quantization Fusion Pass Running ...       Finished.
[04:06:09] PPQ Parameter Quantization Pass Running ...    Finished.
[04:06:09] PPQ Runtime Calibration Pass Running ...       

Calibration Progress(Phase 1): 100%|██████████| 16/16 [01:53<00:00,  7.10s/it]


Finished.
[04:08:03] PPQ LSQ Optimization Running ...               
Check following parameters:
Is Scale Trainable:        True
Interested Layers:         []
Num of blocks:             61
Learning Rate:             1e-05
Steps:                     50
Gamma:                     0.0

# Block [1 / 61]: [node_conv2d -> node_Split_152]


# Tuning Procedure : 100%|██████████| 50/50 [00:03<00:00, 14.61it/s]


# Tuning Finished  : (0.6429 -> 0.5848) [Block Loss]

# Block [2 / 61]: [node_conv2d_3 -> node_conv2d_4/Swish]


...

# Tuning Finished  : (0.9214 -> 0.8620) [Block Loss]

# Block [60 / 61]: [node_conv2d_118 -> node_conv2d_120]


# Tuning Procedure : 100%|██████████| 50/50 [00:00<00:00, 158.40it/s]


# Tuning Finished  : (0.2301 -> 0.1945) [Block Loss]

# Block [61 / 61]: [node_conv2d_121 -> node_conv2d_125]


# Tuning Procedure : 100%|██████████| 50/50 [00:00<00:00, 83.04it/s]


# Tuning Finished  : (0.6123 -> 0.5861) [Block Loss]

Finished.
[05:30:04] PPQ Passive Parameter Quantization Running ... Finished.
[05:30:04] PPQ Quantization Alignment Pass Running ...    Finished.
[05:30:04] ESPDL LUT Fusion Pass Running ...              Finished.


In [ ]:
print("Slicing output Concat nodes into 6 discrete tensors...")

# 1. Remove Aux Heads
output_names = list(graph.outputs.keys())
if len(output_names) >= 6:
    for name in output_names[0:3]:
        if name in graph.outputs: graph.outputs.pop(name)
    prune_graph_safely(graph)

# 2. Slice the Concat into Box/Cls
targets = ["one2one_p3", "one2one_p4", "one2one_p5"]
collected_outputs = {}
for target_name in targets:
    if target_name in graph.outputs:
        original_output_var = graph.variables[target_name]
        producer = original_output_var.source_op

        if producer and producer.type == "Concat":
            box_var, cls_var = None, None
            for input_var in producer.inputs:
                dims = input_var.shape
                if dims is not None:
                    if 4 in dims: box_var = input_var
                    elif model_meta['nc'] in dims: cls_var = input_var

            if box_var and cls_var:
                pair_config = [
                    (box_var, f"{target_name}_box"),
                    (cls_var, f"{target_name}_cls"),
                ]
                for var, new_name in pair_config:
                    old_name = var.name
                    if old_name in graph.variables: graph.variables.pop(old_name)
                    var._name = new_name
                    graph.variables[new_name] = var
                    collected_outputs[new_name] = var

                graph.outputs.pop(target_name)
                graph.remove_operation(producer, keep_coherence=False)
                for var in producer.inputs:
                    if producer in var.dest_ops: var.dest_ops.remove(producer)

# 3. Enforce precise output order matching ESPdl C++ expectations
final_output_list = [
    "one2one_p3_box", "one2one_p3_cls",
    "one2one_p4_box", "one2one_p4_cls",
    "one2one_p5_box", "one2one_p5_cls"
]
graph.outputs.clear()
for name in final_output_list:
    if name in collected_outputs:
        graph.outputs[name] = collected_outputs[name]

prune_graph_safely(graph)
print("✅ Salidas del grafo recortadas y estructuradas correctamente.")

Slicing output Concat nodes into 6 discrete tensors...
✅ Salidas del grafo recortadas y estructuradas correctamente.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# CELL 11: Final Export (Auto-Switching Streaming/Standard)
# ==========================================
import os
import esp_ppq.lib as PFL
# Definir la ruta de destino en Google Drive con extensión .espdl
quantized_model_dir = f"{DRIVE_SAVE_PATH}_Quantized"
os.makedirs(quantized_model_dir, exist_ok=True)
final_espdl_path = os.path.join(
    quantized_model_dir,
    f"{QATConfig.MODEL_NAME}_{QATConfig.IMG_SZ}_s8_{PLATFORM}.espdl"
)
if QATConfig.STREAMING_CHUNK is not None:
    print("Exporting with HardwareAwareEspdlExporter (Streaming enabled)...")
    exporter = PFL.Exporter(platform=QATConfig.TARGET_PLATFORM)
    exporter.export(
        file_path=final_espdl_path,
        graph=graph,
        int16_lut_step=QATConfig.INT16_LUT_STEP
    )
else:
    print("Exporting with Standard PFL.Exporter (Non-Streaming)...")
    exporter = PFL.Exporter(platform=QATConfig.TARGET_PLATFORM)
    exporter.export(final_espdl_path, graph=graph, int16_lut_step=QATConfig.INT16_LUT_STEP)

print(f"Deployment Model exported to {final_espdl_path}")

Exporting with Standard PFL.Exporter (Non-Streaming)...
[ESPDL Exporter] Switching to IDEAL MATH for Table Generation...
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip val_3 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_0 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_4 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_9 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_10 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_7 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_16 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_18 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip softmax because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip PPQ_Variable_2 because it's not exportable
[INFO][ESPDL][2026-09-18 05:30:06]:  Skip 

AttributeError: 'NoneType' object has no attribute 'dtype'

In [ ]:
import os
# La función load_onnx_graph ya no es necesaria aquí, ya que el gráfico se carga en la celda anterior (389e6312).
# from esp_ppq.api.interface import load_onnx_graph

# El graph ya debería estar cargado en la memoria desde la celda anterior (389e6312).
print("Evaluando el modelo cuantificado. Se espera que el 'graph' ya esté cargado desde la celda anterior (389e6312).")

from trainer import QATTrainer
print("Evaluating Target ESP-DL Emulated mAP...")
dummy_trainer = QATTrainer(graph=graph, model_meta=model_meta, device=QATConfig.DEVICE) # 'graph' debe venir de 389e6312
val_mAP = dummy_trainer.eval()
print(f"Final Quantized mAP50-95: {val_mAP:.3f}")

In [ ]:
# ==========================================
# CELL 10.1: Inference Preview (ESP-DL Emulated)
# ==========================================
# Runs the quantized graph on a test image using the same preprocessing
# as the ESP-DL C++ runtime (nearest-neighbour resize + letterbox padding,
# pad_val=114 — matching YOLO26::preprocess() in yolo26.cpp).
#
# Post-processing mirrors YOLO26::postprocess() + YOLO26::decode_grid()
# from esp-dl/models/yolo26/yolo26.cpp exactly:
#   - Sigmoid applied to raw cls scores
#   - Box decoded as: x1=(cx+0.5 - d_l)*stride, x2=(cx+0.5 + d_r)*stride
#   - YOLOv26 one2one head is NMS-FREE: top-K sort by score only
#
# Output filename follows the .espdl export convention:
#   <stem>_<img_sz>_s8_<platform>.jpg   e.g. bus_512_s8_p4.jpg
%matplotlib inline
from notebook_helpers import eval_espdl_model
import os

TEST_IMAGE = os.path.join('/content/drive/MyDrive/Colab Notebooks/Imagenes de Prueba/face_test.jpg')
OUTPUT_DIR = os.path.join('/content/drive/MyDrive/Colab Notebooks/Imagenes de Prueba/Resultado ')

predictions, saved_path = eval_espdl_model(
    test_image_path = TEST_IMAGE,
    graph           = graph,
    target_img_sz   = QATConfig.IMG_SZ,
    data_yaml       = QATConfig.DATA_YAML_FILE,
    platform        = PLATFORM,
    conf_thresh     = 0.25,
    output_dir      = OUTPUT_DIR,
)

print(f"\nDetections ({len(predictions)}):")
for p in predictions:
    print(f"  [{p['class_id']:3d}] {p['class']:20s}  conf={p['score']:.3f}  box={[round(v) for v in p['box']]}")